# **Integration Word2Vec**

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
from sklearn.manifold import TSNE

# Background

## Word2Vec

Word2Vec is a family of methods that transforms words into number vectors, 
grouping words with similar meanings closer together in a mathematical space. 
This lets us measure how related two words are just by looking at the distance 
between their vectors. For example, "pizza" and "burger" sit close together 
since both are fast foods, while a word like "bicycle" lands far away in 
that same space.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0205EN-SkillsNetwork/Words.png" alt="Word2Vec food example" class="bg-primary" width="400px">

In [2]:
corpus = """I traveled to Paris and visited the Eiffel Tower
The backpacker explored hidden trails in the mountains of Nepal
She booked a tiny cozy cabin near the forest for the weekend
The cruise ship sailed across the vast blue Mediterranean Sea
He packed a small bag and headed to the airport with excitement
The local market was filled with colorful spices and exotic fruits
She hiked a long steep trail to reach the breathtaking waterfall
The hostel had a warm friendly atmosphere full of fellow travelers
The ancient temple stood tall and mysterious in the jungle
He rented a small motorbike to explore the winding coastal roads
The beach was peaceful and quiet during the early morning hours
She captured stunning photographs of the golden sunset over the desert
The tiny island was surrounded by crystal clear turquoise waters
He felt a deep sense of freedom as the plane lifted off the runway
The mountain village was cold and remote but incredibly beautiful
She tried the local street food and was amazed by the bold flavors
The train journey through the countryside was scenic and relaxing
He met a kind stranger who showed him the hidden gems of the city
The safari was an incredible experience filled with wild animals
She wandered through narrow cobblestone streets in an old European town
The camping site was nestled between towering pine trees and a river
He climbed a massive sand dune and watched the sun rise over the horizon
The underwater world was filled with colorful coral reefs and fish
She stayed in a luxurious overwater bungalow in the Maldives
The road trip covered thousands of miles across open desert plains
He discovered a small charming cafe hidden in a quiet alley
The volcano hike was challenging but the view from the top was magnificent
She floated peacefully on a wooden boat through the calm river delta
The ancient ruins told stories of a once great and powerful civilization
He zipped through the forest on a thrilling canopy zipline adventure
The mountain pass was narrow and dangerous but the scenery was spectacular
She danced with locals at a vibrant cultural festival in the town square
The lighthouse stood tall and lonely at the edge of the rocky cliff
He explored a massive underground cave system with towering stalactites
The coastal town had small colorful fishing boats lined along the harbor
She felt tiny standing beneath the enormous sequoia trees in the forest
The desert night sky was brilliant with millions of glittering stars
He joined a small guided tour through the winding alleyways of the medina
The glacier was a stunning shade of deep blue and stretched for miles
She took a long peaceful walk along the empty white sandy beach
The waterfall roared loudly as it crashed into the pool far below
He found a quiet spot on the hilltop to read and enjoy the fresh air
The old bridge arched gracefully over the wide and gentle flowing river
She watched the northern lights dance across the dark Arctic sky
The busy airport buzzed with travelers heading to distant corners of the world
He paddled a small kayak through the mangrove forests at sunrise
The historic city center was packed with grand buildings and monuments
She hiked through dense rainforest and discovered a hidden lagoon
The mountain lake was perfectly still reflecting the snow capped peaks above
He explored the vast open plains and watched a herd of elephants roam freely"""

### **Step 1: Tokenization**

In [3]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# define the tokenizer
tokenizer = get_tokenizer("basic_english")
tokenized_corpus = tokenizer(corpus)

print(tokenized_corpus)

# define yield function for tokenization
def yield_tokens (dataset):
    for data in dataset:
        token = tokenizer(data) 
        yield token

corpus_map = map(tokenizer,corpus.split())

#vocabulary = build_vocab_from_iterator(yield_tokens(tokenized_corpus), specials=["<unk>"])
vocabulary = build_vocab_from_iterator(map(tokenizer,corpus.split()), specials=["<unk>"])
vocabulary.set_default_index(vocabulary["<unk>"])


['i', 'traveled', 'to', 'paris', 'and', 'visited', 'the', 'eiffel', 'tower', 'the', 'backpacker', 'explored', 'hidden', 'trails', 'in', 'the', 'mountains', 'of', 'nepal', 'she', 'booked', 'a', 'tiny', 'cozy', 'cabin', 'near', 'the', 'forest', 'for', 'the', 'weekend', 'the', 'cruise', 'ship', 'sailed', 'across', 'the', 'vast', 'blue', 'mediterranean', 'sea', 'he', 'packed', 'a', 'small', 'bag', 'and', 'headed', 'to', 'the', 'airport', 'with', 'excitement', 'the', 'local', 'market', 'was', 'filled', 'with', 'colorful', 'spices', 'and', 'exotic', 'fruits', 'she', 'hiked', 'a', 'long', 'steep', 'trail', 'to', 'reach', 'the', 'breathtaking', 'waterfall', 'the', 'hostel', 'had', 'a', 'warm', 'friendly', 'atmosphere', 'full', 'of', 'fellow', 'travelers', 'the', 'ancient', 'temple', 'stood', 'tall', 'and', 'mysterious', 'in', 'the', 'jungle', 'he', 'rented', 'a', 'small', 'motorbike', 'to', 'explore', 'the', 'winding', 'coastal', 'roads', 'the', 'beach', 'was', 'peaceful', 'and', 'quiet', 'dur

In [5]:
# Test
sample_sentence = "temple stood tall I and mysterious in the jungle"
tokenized_sample = tokenizer(sample_sentence)
encoded_sample = [vocabulary[token] for token in tokenized_sample]
print("Encoded sample:", encoded_sample)

Encoded sample: [282, 57, 59, 179, 3, 212, 9, 1, 187]


In [6]:
# generate a list of vocab indices according to the tokenize corpus
text_pipeline = lambda x: vocabulary(x)
text_pipeline(tokenized_corpus)[0:10]

[179, 293, 12, 222, 3, 302, 1, 131, 289, 1]

In [7]:
indices_to_token = vocabulary.get_itos()
indices_to_token[222]

'paris'

### **Step 2: Context and Target Data**

In [8]:
cbow_data = []
context_size = 2

for i in range(context_size, len(tokenized_corpus)-context_size):
    
    context = (
        [tokenized_corpus[i-context_size+j] for j in range(context_size)] +
        [tokenized_corpus[i+context_size-j] for j in range(context_size)]
    )
    
    target = tokenized_corpus[i]

cbow_data.append((context,target))
    

### **Step 3: Data Loader**

In [9]:
from torch.utils.data import DataLoader

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
def collate_func(batch):
    
    target_list, context_list, offsets = [],[],[0]
    
    for _context, _target in batch:
        
        target_list.append(vocabulary[_target])
        process_context = torch.tensor(text_pipeline(_context), dtype=torch.int64)
        context_list.append(process_context)
        
        offsets.append(process_context.size(0))
    
    target_list = torch.tensor(target_list, dtype=torch.int64)
    offsets = torch.tensor(offsets, dtype=torch.int64)
    context_list = torch.cat(context_list)
    
    return target_list.to(device), context_list.to(device), offsets.to(device) 
    

In [12]:
target_list, context_list, offsets = collate_func(cbow_data[0:10])
print(f"target_list(Tokenized target words): {target_list} , context_list(Surrounding context words): {context_list} , offsets(Starting indexes of context words for each target): {offsets} ")


target_list(Tokenized target words): tensor([132]) , context_list(Surrounding context words): tensor([171,   5, 153, 243]) , offsets(Starting indexes of context words for each target): tensor([0, 4]) 


In [13]:
batch_size = 64

In [14]:
data_loader = DataLoader(
    cbow_data,batch_size=batch_size, shuffle=True, collate_fn=collate_func
)

## **Continuous Bag of Words (CBOW)**
For the Continuous Bag of Words (CBOW) model, we use a "context" to predict a target word. The "context" is typically a set of surrounding words. For example, if your context window is of size 2, then you take two words before and two words after the target word as context. The sentence used here is "She hiked a long steep trail to reach the waterfall". The target word is in <span style="color:red;">red</span> and the context is in <span style="color:blue;">blue</span>:

<table border="1">
    <tr>
        <th>Time Step</th>
        <th>Phrase</th>
    </tr>
    <tr>
        <td>1</td>
        <td><span style="color:blue;">She hiked</span> <span style="color:red;">a</span> <span style="color:blue;">long steep</span></td>
    </tr>
    <tr>
        <td>2</td>
        <td><span style="color:blue;">hiked a</span> <span style="color:red;">long</span> <span style="color:blue;">steep trail</span></td>
    </tr>
    <tr>
        <td>3</td>
        <td><span style="color:blue;">a long</span> <span style="color:red;">steep</span> <span style="color:blue;">trail to</span></td>
    </tr>
    <tr>
        <td>4</td>
        <td><span style="color:blue;">long steep</span> <span style="color:red;">trail</span> <span style="color:blue;">to reach</span></td>
    </tr>
    <tr>
        <td>5</td>
        <td><span style="color:blue;">steep trail</span> <span style="color:red;">to</span> <span style="color:blue;">reach the</span></td>
    </tr>
    <tr>
        <td>6</td>
        <td><span style="color:blue;">trail to</span> <span style="color:red;">reach</span> <span style="color:blue;">the waterfall</span></td>
    </tr>
    <tr>
        <td>7</td>
        <td><span style="color:blue;">to reach</span> <span style="color:red;">the</span> <span style="color:blue;">waterfall</span></td>
    </tr>
</table>

You can slide over the sequence and create training data:


In [15]:
class CBoWModel (nn.Module):
    def __init__(self, context_size, embedded_dim, vocab_size):
        super(CBoWModel,self).__init__()
        self.context_size = context_size
        self.embedded_dim = embedded_dim
        self.vocab_size = vocab_size

        # define the input layer
        self.embeddeding = nn.Embedding(vocab_size, embedded_dim,sparse=False)
        # define the hidden layer
        self.linear_hidden = nn.Linear(embedded_dim, embedded_dim//2)
        #define the output layer
        self.linear_out = nn.Linear(embedded_dim//2, vocab_size )
        
         # Initialize the weights of the model's parameters
        self.init_weights()
        
    
    def init_weights(self):
        
        weight_range = 0.5
        
        # input layer neurones weights to hidden layer
        self.embeddeding.weight.data.uniform_(-weight_range, weight_range)
        # hidden layer neurones weights to output layer
        self.linear_hidden.weight.data.uniform_(-weight_range,weight_range)
        self.linear_out.weight.data.zero_()
    
    def forward(self, text, offsets):
        out_embedding = self.embeddeding(text, offsets)
        
        out_linear_hidden = torch.relu(self.linear_hidden(out_embedding))
        
        return self.linear_out(out_linear_hidden)

Create an instance of the CBOW model:


In [16]:
context_size = 2
embedded_dim = 24
vocab_size = len(vocabulary)

In [17]:
cbow_model = CBoWModel(context_size,embedded_dim,vocab_size)

In [18]:
print(cbow_model.named_modules())

<generator object Module.named_modules at 0x000002C63D2EB0A0>


Define the loss function, optimizer, and scheduler for training:


In [19]:
learning_rate = 5

#define loss function
criterion = nn.CrossEntropyLoss()

# define optimizer function
optimizer = torch.optim.SGD(cbow_model.parameters(),lr=learning_rate)

# define scheduler function
scheduler = torch.optim.lr_scheduler.StepLR(optimizer,1.0,gamma=0.1)

Define Model Training

In [20]:
from tqdm import tqdm

In [21]:
def train_model(model, dataloader, criterion, optimizer, num_epochs = 1000):
    
    average_loss = []
    
    for epoch in tqdm(range(num_epochs)):
        
        epoch_loss = 0
        
        
         # Using tqdm for a progress bar
        for idx, samples in enumerate(dataloader):
            
            # optimizer should be at zero grad initially
            optimizer.zero_grad()
             
            #check embedding bag layer instance
            if any(isinstance(nn.EmbeddingBag) for _,module in model.named_modules()):
                target, context, offsets = samples
                predicted = model(context, offsets)
            
            elif any(isinstance(nn.Embedding) for _, module in model.named_modules()):
                  target, context = samples
                  predicted = model(context)
            
            loss = criterion(predicted, target)
            loss.backward()
            # prevent grad being exploading
            nn.utils.clip_grad_norm(model.parameters(), 0.1)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        average_loss.append(epoch_loss/len(dataloader))
    return model, average_loss
    

### Train the Model


In [ ]:
model, average_loss = train_model(cbow_model,data_loader,criterion,optimizer,num_epochs=400)